In [15]:
!pip install tensorflow


In [ ]:
import tensorflow as tf
import numpy as np

# 1. إعداد البيانات لتدريب نموذج Keras
X = np.array([[1, 1, 1, 0], [0, 1, 1, 0], [1, 0, 1, 1], [0, 0, 1, 1]])  # الأعراض
y = np.array([0, 1, 2, 2])  # التشخيص (0: إنفلونزا، 1: صداع نصفي، 2: تسمم غذائي)

# 2. تدريب نموذج Keras
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_dim=4),
    tf.keras.layers.Dense(3, activation='softmax')  # 3 مخرجات للتشخيصات الثلاثة
])

# 3. تجميع النموذج
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 4. تدريب النموذج
model.fit(X, y, epochs=100)

# 5. حفظ النموذج
model.save('medical_diagnosis_model.h5')

# 6. تحويل النموذج إلى TFLite باستخدام TensorFlow
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 7. حفظ النموذج المحول إلى TFLite
with open('medical_diagnosis_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ تم تدريب النموذج وتحويله إلى TFLite!")

# تحميل النموذج المحول إلى TFLite
interpreter = tf.lite.Interpreter(model_path="medical_diagnosis_model.tflite")
interpreter.allocate_tensors()

# الحصول على المدخلات والمخرجات من النموذج
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# تعريف الأعراض مع الأسماء
symptom_names = ["ارتفاع درجة الحرارة", "سعال", "صداع", "إسهال"]

# دالة لتلقي المدخلات من المستخدم مع عرض الأعراض
def get_symptoms():
    print("أدخل الأعراض (0 = لا، 1 = نعم):")

    symptoms = []

    for i, symptom in enumerate(symptom_names):
        print(f"{i+1}. {symptom}")
        symptom_value = int(input(f"هل تعاني من {symptom}? (0 = لا، 1 = نعم): "))
        symptoms.append(symptom_value)

    return np.array([symptoms], dtype=np.float32)

# دالة لاختبار النموذج المحول وتنبؤ التشخيص
def diagnose():
    symptoms = get_symptoms()  # الحصول على الأعراض من المستخدم

    # تمرير المدخلات إلى النموذج
    interpreter.set_tensor(input_details[0]['index'], symptoms)

    # تنفيذ التنبؤ
    interpreter.invoke()

    # الحصول على النتيجة المتوقعة
    output_data = interpreter.get_tensor(output_details[0]['index'])
    diagnosis = np.argmax(output_data[0])  # الحصول على التشخيص باستخدام القيمة الأكبر

    # عرض التشخيص
    if diagnosis == 0:
        result = "إنفلونزا"
    elif diagnosis == 1:
        result = "صداع نصفي"
    elif diagnosis == 2:
        result = "تسمم غذائي"
    else:
        result = "تشخيص غير معروف"

    print(f"التشخيص: {result}")

# استدعاء دالة التشخيص
diagnose()


Epoch 1/100


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step - accuracy: 0.7500 - loss: 0.8471
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.7500 - loss: 0.8405
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7500 - loss: 0.8340
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.7500 - loss: 0.8276
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7500 - loss: 0.8213
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7500 - loss: 0.8150
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7500 - loss: 0.8088
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.7500 - loss: 0.8027
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.7500 - loss: 0.7967
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.7500 - loss: 0.7907
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.7500 - loss: 0.7848
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7500 - loss: 0.7791


Saved artifact at '/tmp/tmp_1xfy1m2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 4), dtype=tf.float32, name='keras_tensor_17')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  134934700463856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134934700467552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134934700462096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134934700463328: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ تم تدريب النموذج وتحويله إلى TFLite!
أدخل الأعراض (0 = لا، 1 = نعم):
1. ارتفاع درجة الحرارة
